# Session Timing Audit — AffectAI Data Processing

**Purpose:** Comprehensive documentation of task and phase timing for all sessions,
including discussion-only durations, T0 free_talk baseline windows, data completeness,
and quality flags.

**Data sources:**
- `beh/*_task-T{0..4}_run-01_events.tsv` — per-task stimuli events with phase markers (LSL-timestamped)
- `annot/*_task_run_windows.tsv` — task-level boundaries (wall clock + LSL)
- `metadata/session_metadata_report.tsv` — session-level metadata

**Extraction script:** `tools/features/extract_session_timing.py`

## Task & Phase Structure

Each session runs 5 tasks in fixed order. Within each task, phases are pushed by the
stimuli server and recorded as `push_content` events with `"phase": "<name>"`.

| Task | Phases (in order) | Discussion window |
|------|-------------------|-------------------|
| **T0** Baseline/Intro | welcome → study_introduction → vad_introduction → postblock_introduction → **free_talk** → finish | free_talk → finish |
| **T1** Hidden-Profile Decision | tobii_calibration → intro → evidence_card (silent reading, 75s) → **discussion_selection** → candidate_selection (60s) → finish | discussion_selection → finish |
| **T2** Mini-Negotiation | tobii_calibration → shared_brief → **role_card** → settlement_form → finish | role_card → finish |
| **T3** Idea Generation (NGT) | tobii_calibration → instructions → idea_generation (silent, ~150–180s) → **show_ideas_discussion** → group_selection (60s) → finish | show_ideas_discussion → finish |
| **T4** Public-Goods Game | tobii_calibration → shared_brief → contribution_form → **discussion** → finish | discussion → finish |

**Discussion window** = the phase where participants actively discuss (bold phase → finish).
This excludes calibration, instruction reading, silent phases, and form-filling.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

REPO = Path.cwd()
TIMING_DIR = REPO / 'analysis' / 'results' / 'session_timing'

task_df = pd.read_csv(TIMING_DIR / 'task_timing.tsv', sep='\t')
phase_df = pd.read_csv(TIMING_DIR / 'phase_timing.tsv', sep='\t')
comp_df = pd.read_csv(TIMING_DIR / 'completeness_summary.tsv', sep='\t')

print(f'Task timing: {task_df.shape[0]} rows ({task_df.group_id.nunique()} groups × {task_df.task.nunique()} tasks)')
print(f'Phase timing: {phase_df.shape[0]} rows')
print(f'Sessions: {comp_df.shape[0]}')

## 1. Session Overview — Dates, Times, and Data Completeness

In [ ]:
display(comp_df[['session_id', 'group_id', 'session_date', 'session_datetime_utc',
                  'n_tasks_with_events', 'n_tasks_with_windows',
                  'n_discussion_durations', 'has_free_talk_baseline', 'quality_flags']]
        .rename(columns={'n_tasks_with_events': 'events', 'n_tasks_with_windows': 'windows',
                         'n_discussion_durations': 'disc_dur', 'has_free_talk_baseline': 'free_talk'})
        .to_string(index=False))

n_complete = (comp_df['n_tasks_with_events'] == 5).sum()
n_ft = (comp_df['has_free_talk_baseline'] == True).sum()
print(f'\n{n_complete}/10 sessions have complete events for all 5 tasks')
print(f'{n_ft}/10 sessions have free_talk baseline')

## 2. Task Durations (Total Block)

Duration from the first stimuli event to the `finish` marker. Includes all phases
(calibration, instructions, reading, discussion, forms).

In [ ]:
# Pivot: group × task → duration
dur_pivot = task_df.pivot(index='group_id', columns='task', values='task_duration_s')
dur_pivot = dur_pivot[['T0', 'T1', 'T2', 'T3', 'T4']]
dur_pivot_min = dur_pivot / 60

print('Task durations (minutes):')
display(dur_pivot_min.round(1))

# Flag anomalies
print('\n--- Anomalies ---')
for grp in dur_pivot.index:
    for task in dur_pivot.columns:
        val = dur_pivot.loc[grp, task]
        if pd.notna(val) and val > 3600:
            print(f'  {grp} {task}: {val:.0f}s ({val/3600:.1f}h) — overnight clock issue')
        elif pd.isna(val):
            print(f'  {grp} {task}: MISSING')

## 3. Discussion-Only Durations

The discussion window starts when the active discussion phase is pushed and ends at `finish`.
This is the **correct** duration for analysis — it excludes calibration, silent reading,
silent idea generation, and instruction phases.

| Task | Discussion starts at | What precedes it |
|------|---------------------|------------------|
| T1 | `discussion_selection` | Silent evidence card reading (75s timer) |
| T2 | `role_card` | Shared brief reading |
| T3 | `show_ideas_discussion` | Silent idea generation (~150–180s) |
| T4 | `discussion` | Private contribution form (60s) + outcome reveal |

**Important for T2:** `role_card` marks when roles are displayed, but actual verbal
discussion may start a few seconds later. The transcript files capture the true
speech onset, which would provide a more precise discussion start.

In [ ]:
disc = task_df[task_df['discussion_duration_s'].notna() & (task_df['task'] != 'T0')].copy()

# Exclude grp-10 T0 overnight anomaly
disc = disc[~((disc['group_id'] == 'grp-10') & (disc['discussion_duration_s'] > 3600))]

disc_pivot = disc.pivot(index='group_id', columns='task', values='discussion_duration_s')
disc_pivot = disc_pivot[['T1', 'T2', 'T3', 'T4']]
disc_pivot_min = disc_pivot / 60

print('Discussion-only durations (minutes):')
display(disc_pivot_min.round(1))

print('\nSummary statistics (minutes):')
display(disc_pivot_min.describe().round(1))

## 4. T0 Free Talk Baseline Window

The `free_talk` phase within T0 is the recommended baseline for pupil diameter correction.
During this phase, participants engage in unstructured casual conversation — no task demands,
no instructions, no cognitive load from the experiment.

In [ ]:
ft = task_df[(task_df['task'] == 'T0')][['group_id', 'session_date', 'task_duration_s',
                                          'discussion_onset_s', 'discussion_duration_s',
                                          'quality_notes']].copy()
ft.columns = ['group_id', 'date', 't0_total_s', 'free_talk_onset_s', 'free_talk_duration_s', 'notes']

# Flag the overnight issue
ft.loc[ft['free_talk_duration_s'] > 3600, 'notes'] = 'overnight clock issue; duration unreliable'

print('T0 Free Talk baseline windows:')
display(ft.to_string(index=False))

valid_ft = ft[(ft['free_talk_duration_s'].notna()) & (ft['free_talk_duration_s'] < 3600)]
print(f'\nUsable free_talk baselines: {len(valid_ft)}/10 sessions')
print(f'Duration range: {valid_ft["free_talk_duration_s"].min():.0f}–{valid_ft["free_talk_duration_s"].max():.0f}s '
      f'({valid_ft["free_talk_duration_s"].min()/60:.1f}–{valid_ft["free_talk_duration_s"].max()/60:.1f} min)')
print(f'Mean: {valid_ft["free_talk_duration_s"].mean():.0f}s ({valid_ft["free_talk_duration_s"].mean()/60:.1f} min)')

## 5. Phase-Level Timing Detail

Complete phase sequence for each session × task, with onset and duration.

In [ ]:
# Show phase detail for one representative session
example_grp = 'grp-07'
print(f'Phase timing for {example_grp} (all tasks):\n')
ex = phase_df[phase_df['group_id'] == example_grp][['task', 'phase', 'onset_s', 'duration_s']].copy()
ex['duration_min'] = (ex['duration_s'] / 60).round(1)
display(ex.to_string(index=False))

In [ ]:
# Cross-session phase duration statistics
print('Phase duration statistics across sessions (seconds):\n')
phase_stats = phase_df.groupby(['task', 'phase'])['duration_s'].agg(['count', 'mean', 'std', 'min', 'max'])
phase_stats = phase_stats.round(1)
display(phase_stats)

## 6. Data Completeness & Missingness

### Per-task events availability

In [ ]:
events_pivot = task_df.pivot(index='group_id', columns='task', values='has_events')
events_pivot = events_pivot[['T0', 'T1', 'T2', 'T3', 'T4']]
print('Events data availability (True = has phase-level events):')
display(events_pivot)

windows_pivot = task_df.pivot(index='group_id', columns='task', values='has_task_window')
windows_pivot = windows_pivot[['T0', 'T1', 'T2', 'T3', 'T4']]
print('\nTask window availability (True = has LSL-aligned boundaries):')
display(windows_pivot)

In [ ]:
# Missing phases per session
missing = task_df[task_df['missing_phases'].notna() & (task_df['missing_phases'] != '')]
if len(missing):
    print('Sessions with missing expected phases:')
    display(missing[['group_id', 'task', 'missing_phases', 'quality_notes']].to_string(index=False))
else:
    print('No missing phases in sessions with events data.')

## 7. Quality Issues & Known Problems

In [ ]:
issues = task_df[task_df['quality_notes'].notna() & (task_df['quality_notes'] != '')]
if len(issues):
    print('Quality issues:')
    display(issues[['group_id', 'task', 'quality_notes']].to_string(index=False))
else:
    print('No quality issues flagged.')

print('\n--- Summary of known issues ---')
print('• grp-08: events.tsv files empty for all tasks; timing recovered from recording-lsl_events.tsv')
print('• grp-09: T0 missing entirely (no events, no task window); T1 has only 23.6s — likely incomplete')
print('• grp-10: T0 block duration 70018s (19.4h) due to overnight stimuli app left running')
print('  → Free talk onset at 725.6s into T0 is valid; finish timestamp is not')
print('  → T1–T4 discussion durations are valid (normal range)')
print('• grp-11: T0 events empty; no recording-lsl file; T0 data missing (documented in data_audit.md)')
print('• grp-14: T0 finish marker missing in events.tsv; used task_run_windows end time')

## 8. Complete Timing Reference Table

Full table for thesis appendix: every group × task with block duration, discussion duration,
and session date/time.

In [ ]:
ref = task_df[['group_id', 'session_date', 'task', 'task_duration_s',
               'discussion_onset_s', 'discussion_duration_s', 'quality_notes']].copy()

# Exclude grp-10 T0 anomalous discussion duration
ref.loc[(ref['group_id'] == 'grp-10') & (ref['task'] == 'T0') & (ref['discussion_duration_s'] > 3600),
        'discussion_duration_s'] = np.nan

ref['block_min'] = (ref['task_duration_s'] / 60).round(1)
ref['disc_min'] = (ref['discussion_duration_s'] / 60).round(1)

print('Complete timing reference (for thesis appendix):')
display(ref[['group_id', 'session_date', 'task', 'block_min', 'disc_min', 'quality_notes']]
        .to_string(index=False))

In [ ]:
# Save complete reference table
out_path = TIMING_DIR / 'thesis_timing_reference.tsv'
ref[['group_id', 'session_date', 'task', 'task_duration_s', 'block_min',
     'discussion_onset_s', 'discussion_duration_s', 'disc_min', 'quality_notes']].to_csv(
    out_path, sep='\t', index=False)
print(f'Saved: {out_path}')